[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/066.%20The%20Workforce%20Scheduling%20for%20Peak_Off-Peak%20Problem/P66-Tier-5_executed.ipynb)

---


# 66. The Workforce Scheduling for Peak/Off-Peak Problem
## Tier 5: Workforce Simulation and Scenario Analysis

### Purpose
This notebook builds a small, discrete-time workforce simulation and uses it to compare fixed employee schedules under different customer-arrival patterns. It illustrates how a simulation can support operational planning by making assumptions explicit and reporting service, waiting, staffing, and cost measures.

The model uses synthetic customer arrivals and a simplified service process. It does not find an optimal schedule: it evaluates the schedules supplied to it. The example is a teaching prototype, not a live digital twin.

### Inputs and outputs
- **Inputs:** Employee and workstation descriptions, sensor descriptions, candidate shift schedules, an arrival pattern, and simulation settings such as duration and time-step length.
- **Outputs:** Scenario-level metrics, a comparison table, visual summaries, and a status report for the simulated system.
- **Main workflow:** Define the model → initialize its entities → define schedules → simulate schedule-and-demand scenarios → compare results → inspect illustrative monitoring and what-if examples.

### Model assumptions and scope
- Simulation time and shift boundaries are measured in minutes. Employee pay rates are measured in dollars per hour, and waiting cost is modeled at \$10 per waiting customer-hour.
- Customers arrive according to a selected synthetic pattern. Their service requirements are recorded but do not change the service process or service duration.
- Each employee can serve at most one customer at a time. Service duration is sampled when a customer begins service. A service already in progress may finish after the employee's scheduled shift ends; that time is counted as paid overtime.
- Workstation and sensor objects are stored as model entities, but they do not constrain service capacity or receive live readings. The simulation does not connect to physical operations, forecast demand from external data, or generate statistical confidence intervals.
- A run is one stochastic simulation realization. Because scenarios consume random draws in sequence, scenario comparisons are descriptive rather than paired statistical experiments.
- The candidate schedules are examples for comparison, not solutions produced by an optimization algorithm. No schedule optimization is performed.

### Simulation logic
At each time step, the model identifies scheduled employees, records customers who arrived during the interval, assigns waiting customers to available employees, and accumulates labor and waiting costs. An employee serving a customer remains occupied until that service is complete.

Let \(t\) be the start of a time step, \(\Delta t\) its duration, and \(T\) the simulation horizon. An employee \(e\) is scheduled at time \(t\) when their shift starts no later than \(t\) and ends after \(t\). A customer \(c\) is waiting when they have arrived and have not yet been assigned to service. If \(A_t\) is the set of available employees and \(Q_t\) is the set of waiting customers, the number assigned at that step is

$$
N_t = \min\bigl(|A_t|, |Q_t|\bigr).
$$

A customer assigned at time \(t\) starts service at that time and occupies one employee for their sampled service duration. The step's labor cost is the hourly pay of employees working during the interval, multiplied by the interval's duration in hours. Its waiting cost is \$10 multiplied by the number of customers still waiting and the interval's duration in hours.

The reported service level is the proportion of completed customers whose waiting time does not exceed their individual patience threshold. Average waiting time is computed over completed customers. Employee utilization is the total service time divided by paid employee time, including overtime for service that extends beyond a shift. These definitions connect the simulation's event logic to the metrics reported in the comparison table.

The simulation evaluates \(\lceil T/\Delta t\rceil\) time steps. At each step it checks the customer and employee lists, so computational work grows with the number of time steps and modeled entities. This implementation is intended for small illustrative scenarios, not large-scale operational simulation.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Fixed seeds make the example reproducible when run from a fresh kernel.
random.seed(42)
np.random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

## 1. Represent the simulated entities

The data classes below give each modeled entity a clear set of attributes. Employees carry their skills and pay rates; customers carry their arrival, service, and patience information; workstations and sensors describe additional operational resources. The service model uses employees as its capacity constraint. Workstations and sensors are represented so that the system's scope is visible, but the current simulation does not use them to route customers or ingest external readings.

In [ ]:
@dataclass
class Employee:
    """An employee represented in the workforce simulation."""

    id: int
    name: str
    skills: List[str]
    hourly_rate: float
    max_hours: int
    current_location: str = "break_room"
    current_task: Optional[str] = None
    fatigue_level: float = 0.0
    satisfaction_score: float = 0.8
    shift_start: Optional[float] = None
    shift_end: Optional[float] = None


@dataclass
class Customer:
    """A customer with an arrival, service, and patience record."""

    id: int
    arrival_time: float
    service_requirement: str
    patience_level: float
    service_time: float
    wait_start_time: Optional[float] = None
    service_start_time: Optional[float] = None
    departure_time: Optional[float] = None
    satisfied: bool = True
    assigned_employee_id: Optional[int] = None


@dataclass
class Workstation:
    """A workstation description stored by the simulation."""

    id: int
    name: str
    capacity: int
    current_load: int = 0
    processing_efficiency: float = 1.0
    maintenance_status: str = "operational"
    queue_length: int = 0


@dataclass
class IoTSensor:
    """A sensor description; this prototype does not read live sensor data."""

    id: int
    name: str
    sensor_type: str
    location: str
    reading_frequency: float
    accuracy: float
    last_reading: Optional[float] = None
    calibration_status: str = "calibrated"

## 2. Build the simulation engine

The `WorkforceDigitalTwin` class stores the entities, generates synthetic arrivals, applies a schedule, advances the simulation, and summarizes outcomes. Here, “digital twin” names the simulated representation of the workforce system; the implementation is not synchronized with a physical operation.

The key state is the information needed to continue the simulation: current time, customer service status, employee shifts, and accumulated performance measures. Retaining this state between time steps is necessary because a customer may still be waiting or receiving service at the next step. In the code, these values are held on the class instance and updated by `simulate_step` and `run_simulation`.

In [ ]:
class WorkforceDigitalTwin:
    """A discrete-time simulation for workforce and customer-flow scenarios."""

    def __init__(self, simulation_duration: float = 480.0, time_step: float = 1.0):
        if simulation_duration <= 0:
            raise ValueError("Simulation duration must be positive.")
        if time_step <= 0:
            raise ValueError("Time step must be positive.")

        self.simulation_duration = float(simulation_duration)
        self.active_simulation_duration = float(simulation_duration)
        self.time_step = float(time_step)
        self.current_time = 0.0

        self.employees: List[Employee] = []
        self.customers: List[Customer] = []
        self.workstations: List[Workstation] = []
        self.sensors: List[IoTSensor] = []

        self.current_schedule: Dict[int, List[float]] = {}
        self.task_assignments: Dict[int, List[str]] = {}
        self.performance_history: List[Dict] = []
        self.kpi_tracker: Dict[str, List[float]] = {
            "service_level": [],
            "avg_wait_time": [],
            "employee_utilization": [],
            "customer_satisfaction": [],
            "operational_cost": [],
            "throughput": [],
            "paid_employee_minutes": [],
            "busy_employee_minutes": [],
        }

        # False indicates that no live external data source is connected.
        self.sync_status: Dict[str, bool] = {
            "employee_data": False,
            "customer_flow": False,
            "workstation_status": False,
            "sensor_readings": False,
        }
        self.scenario_results: Dict[str, Dict] = {}

    def initialize_employees(self, employee_configs: List[Dict]) -> None:
        """Create employees from configuration dictionaries."""
        self.employees = []
        for config in employee_configs:
            self.employees.append(
                Employee(
                    id=config["id"],
                    name=config["name"],
                    skills=config["skills"],
                    hourly_rate=config["hourly_rate"],
                    max_hours=config["max_hours"],
                    satisfaction_score=config.get("satisfaction_score", 0.8),
                )
            )

    def initialize_workstations(self, workstation_configs: List[Dict]) -> None:
        """Create workstation descriptions from configuration dictionaries."""
        self.workstations = []
        for config in workstation_configs:
            self.workstations.append(
                Workstation(
                    id=config["id"],
                    name=config["name"],
                    capacity=config["capacity"],
                    processing_efficiency=config.get("efficiency", 1.0),
                )
            )

    def initialize_sensors(self, sensor_configs: List[Dict]) -> None:
        """Create sensor descriptions without connecting to physical devices."""
        self.sensors = []
        for config in sensor_configs:
            self.sensors.append(
                IoTSensor(
                    id=config["id"],
                    name=config["name"],
                    sensor_type=config["type"],
                    location=config["location"],
                    reading_frequency=config["frequency"],
                    accuracy=config.get("accuracy", 0.95),
                )
            )

    def generate_customer_arrivals(
        self,
        arrival_pattern: str = "variable",
        demand_multiplier: float = 1.0,
    ) -> None:
        """Generate synthetic customer arrivals for the active horizon."""
        valid_patterns = {"constant", "variable", "peak"}
        if arrival_pattern not in valid_patterns:
            raise ValueError(f"Arrival pattern must be one of {sorted(valid_patterns)}.")
        if demand_multiplier < 0:
            raise ValueError("Demand multiplier cannot be negative.")

        self.customers = []
        customer_id = 0
        duration = self.active_simulation_duration

        if arrival_pattern == "constant":
            arrival_rate = 2.0 * demand_multiplier
            if arrival_rate == 0:
                return
            arrival_time = 0.0
            while arrival_time < duration:
                self.customers.append(
                    Customer(
                        id=customer_id,
                        arrival_time=arrival_time,
                        service_requirement=random.choice(
                            ["basic", "standard", "premium"]
                        ),
                        patience_level=random.uniform(0.5, 1.0),
                        service_time=random.uniform(5, 15),
                    )
                )
                customer_id += 1
                arrival_time = customer_id / arrival_rate

        else:
            for minute in range(int(np.ceil(duration))):
                if arrival_pattern == "variable":
                    time_factor = 1.0 + 2.0 * np.sin(
                        2 * np.pi * minute / duration
                    )
                    arrival_rate = max(0.5, time_factor * 1.5)
                else:
                    arrival_rate = 0.5
                    peak_periods = [(120, 240), (300, 420)]
                    if any(start <= minute <= end for start, end in peak_periods):
                        arrival_rate = 4.0

                arrival_rate *= demand_multiplier
                arrivals = np.random.poisson(arrival_rate)
                for _ in range(arrivals):
                    arrival_time = minute + random.random()
                    if arrival_time >= duration:
                        continue

                    if arrival_pattern == "variable":
                        patience = random.uniform(0.3, 1.0)
                        service_time = random.uniform(3, 20)
                    else:
                        patience = random.uniform(0.2, 0.9)
                        service_time = random.uniform(2, 25)

                    self.customers.append(
                        Customer(
                            id=customer_id,
                            arrival_time=arrival_time,
                            service_requirement=random.choice(
                                ["basic", "standard", "premium"]
                            ),
                            patience_level=patience,
                            service_time=service_time,
                        )
                    )
                    customer_id += 1

        self.customers.sort(key=lambda customer: customer.arrival_time)

    def assign_schedule(self, schedule: Dict[int, List[float]]) -> None:
        """Set employee shifts and make the schedule available to the model."""
        employee_ids = {employee.id for employee in self.employees}
        unknown_ids = set(schedule) - employee_ids
        if unknown_ids:
            raise ValueError(f"Schedule contains unknown employee IDs: {unknown_ids}.")

        for employee_id, shift in schedule.items():
            if len(shift) != 2:
                raise ValueError("Each shift must contain a start and end time.")
            shift_start, shift_end = shift
            if shift_start < 0 or shift_end <= shift_start:
                raise ValueError("Each shift must have a nonnegative start and a later end.")

        self.current_schedule = {employee_id: list(shift) for employee_id, shift in schedule.items()}
        for employee in self.employees:
            employee.shift_start = None
            employee.shift_end = None

        for employee_id, (shift_start, shift_end) in self.current_schedule.items():
            employee = next(employee for employee in self.employees if employee.id == employee_id)
            employee.shift_start = shift_start
            employee.shift_end = shift_end

    def simulate_step(self, time: float) -> Dict:
        """Advance customer and employee states over one simulation interval."""
        step_end = min(time + self.time_step, self.active_simulation_duration)
        interval_minutes = max(0.0, step_end - time)
        step_results = {
            "time": time,
            "active_employees": 0,
            "waiting_customers": 0,
            "served_customers": 0,
            "avg_wait_time": 0.0,
            "service_level": 0.0,
            "employee_utilization": 0.0,
            "operational_cost": 0.0,
            "paid_employee_minutes": 0.0,
            "busy_employee_minutes": 0.0,
        }

        active_employees = []
        for employee in self.employees:
            on_shift = (
                employee.shift_start is not None
                and employee.shift_end is not None
                and employee.shift_start <= time < employee.shift_end
            )
            employee.current_location = "workstation" if on_shift else "off_duty"
            if on_shift:
                active_employees.append(employee)

        # Assign arrivals to the interval by their actual arrival time, not by
        # proximity to the interval boundary; this preserves every generated arrival.
        for customer in self.customers:
            if time <= customer.arrival_time < step_end:
                customer.wait_start_time = customer.arrival_time

        busy_employee_ids = {
            customer.assigned_employee_id
            for customer in self.customers
            if customer.assigned_employee_id is not None
            and customer.service_start_time is not None
            and customer.departure_time is not None
            and customer.service_start_time <= time < customer.departure_time
        }
        available_employees = [
            employee
            for employee in active_employees
            if employee.id not in busy_employee_ids
        ]
        waiting_customers = [
            customer
            for customer in self.customers
            if customer.wait_start_time is not None
            and customer.assigned_employee_id is None
        ]

        for employee, customer in zip(available_employees, waiting_customers):
            customer.service_start_time = time
            customer.assigned_employee_id = employee.id
            customer.departure_time = time + customer.service_time
            wait_time = customer.service_start_time - customer.wait_start_time
            customer.satisfied = wait_time <= customer.patience_level * 10.0

        waiting_customers = [
            customer
            for customer in self.customers
            if customer.wait_start_time is not None
            and customer.assigned_employee_id is None
        ]
        step_results["active_employees"] = len(active_employees)
        step_results["waiting_customers"] = len(waiting_customers)
        step_results["served_customers"] = sum(
            1
            for customer in self.customers
            if customer.departure_time is not None
            and time < customer.departure_time <= step_end
        )

        all_working_ids = set(busy_employee_ids)
        all_working_ids.update(employee.id for employee in active_employees)
        employees_by_id = {employee.id: employee for employee in self.employees}
        labor_cost = sum(
            employees_by_id[employee_id].hourly_rate
            for employee_id in all_working_ids
        ) * (interval_minutes / 60.0)
        waiting_cost = len(waiting_customers) * 10.0 * (interval_minutes / 60.0)
        step_results["operational_cost"] = labor_cost + waiting_cost

        paid_employee_minutes = len(all_working_ids) * interval_minutes
        busy_employee_minutes = 0.0
        for customer in self.customers:
            if (
                customer.assigned_employee_id is None
                or customer.service_start_time is None
                or customer.departure_time is None
            ):
                continue
            overlap_start = max(time, customer.service_start_time)
            overlap_end = min(step_end, customer.departure_time)
            if overlap_end > overlap_start:
                busy_employee_minutes += overlap_end - overlap_start

        step_results["paid_employee_minutes"] = paid_employee_minutes
        step_results["busy_employee_minutes"] = busy_employee_minutes
        step_results["employee_utilization"] = (
            busy_employee_minutes / paid_employee_minutes
            if paid_employee_minutes
            else 0.0
        )

        completed_customers = [
            customer
            for customer in self.customers
            if customer.departure_time is not None
            and time < customer.departure_time <= step_end
        ]
        if completed_customers:
            waits = [
                customer.service_start_time - customer.wait_start_time
                for customer in completed_customers
            ]
            step_results["avg_wait_time"] = float(np.mean(waits))
            step_results["service_level"] = float(
                np.mean([customer.satisfied for customer in completed_customers])
            )

        self.current_time = step_end
        return step_results

    def run_simulation(
        self,
        schedule: Dict[int, List[float]],
        arrival_pattern: str = "variable",
        demand_multiplier: float = 1.0,
        simulation_duration: Optional[float] = None,
    ) -> Dict:
        """Run one schedule-and-arrival scenario and return its summary metrics."""
        duration = (
            self.simulation_duration
            if simulation_duration is None
            else float(simulation_duration)
        )
        if duration <= 0:
            raise ValueError("Simulation duration must be positive.")

        self.active_simulation_duration = duration
        self.current_time = 0.0
        self.performance_history = []
        self.kpi_tracker = {key: [] for key in self.kpi_tracker}

        self.assign_schedule(schedule)
        self.generate_customer_arrivals(arrival_pattern, demand_multiplier)

        number_of_steps = int(np.ceil(duration / self.time_step))
        for step in range(number_of_steps):
            step_time = step * self.time_step
            if step_time >= duration:
                break
            step_results = self.simulate_step(step_time)
            self.performance_history.append(step_results)
            self.kpi_tracker["service_level"].append(step_results["service_level"])
            self.kpi_tracker["avg_wait_time"].append(step_results["avg_wait_time"])
            self.kpi_tracker["employee_utilization"].append(
                step_results["employee_utilization"]
            )
            self.kpi_tracker["operational_cost"].append(
                step_results["operational_cost"]
            )
            self.kpi_tracker["throughput"].append(step_results["served_customers"])
            self.kpi_tracker["paid_employee_minutes"].append(
                step_results["paid_employee_minutes"]
            )
            self.kpi_tracker["busy_employee_minutes"].append(
                step_results["busy_employee_minutes"]
            )

        final_results = self._calculate_final_metrics()
        self.active_simulation_duration = self.simulation_duration
        return final_results

    def _calculate_final_metrics(self) -> Dict:
        """Aggregate customer, staffing, and cost measures for the completed run."""
        if not self.performance_history:
            return {}

        completed_customers = [
            customer
            for customer in self.customers
            if customer.departure_time is not None
            and customer.departure_time <= self.active_simulation_duration
        ]
        waits = [
            customer.service_start_time - customer.wait_start_time
            for customer in completed_customers
        ]
        satisfied_count = sum(customer.satisfied for customer in completed_customers)
        total_cost = float(sum(self.kpi_tracker["operational_cost"]))
        paid_minutes = float(sum(self.kpi_tracker["paid_employee_minutes"]))
        busy_minutes = float(sum(self.kpi_tracker["busy_employee_minutes"]))

        return {
            "total_customers": len(self.customers),
            "served_customers": len(completed_customers),
            "satisfied_customers": int(satisfied_count),
            "service_level": (
                satisfied_count / len(completed_customers)
                if completed_customers
                else 0.0
            ),
            "avg_wait_time": float(np.mean(waits)) if waits else 0.0,
            "peak_wait_time": max(waits) if waits else 0.0,
            "avg_utilization": busy_minutes / paid_minutes if paid_minutes else 0.0,
            "total_cost": total_cost,
            "cost_per_customer": (
                total_cost / len(completed_customers)
                if completed_customers
                else 0.0
            ),
            "throughput": len(completed_customers),
            "peak_queue_length": max(
                step["waiting_customers"] for step in self.performance_history
            ),
            "simulation_duration": self.active_simulation_duration,
        }

    def run_scenario_analysis(
        self,
        schedules: Dict[str, Dict[int, List[float]]],
        arrival_patterns: Optional[List[str]] = None,
    ) -> Dict:
        """Evaluate every supplied schedule against each selected arrival pattern."""
        if arrival_patterns is None:
            arrival_patterns = ["variable"]

        self.scenario_results = {}
        for schedule_name, schedule in schedules.items():
            schedule_results = {}
            for pattern in arrival_patterns:
                print(f"Testing scenario: {schedule_name} with {pattern} arrivals...")
                results = self.run_simulation(schedule, pattern)
                self.scenario_results[f"{schedule_name}_{pattern}"] = results
                schedule_results[pattern] = results
            self.scenario_results[schedule_name] = schedule_results

        return self.scenario_results

    def get_real_time_status(self) -> Dict:
        """Report simulated state and indicate whether external feeds are connected."""
        return {
            "simulation_time_minutes": self.current_time,
            "sync_status": dict(self.sync_status),
            "active_employees": sum(
                employee.current_location == "workstation"
                for employee in self.employees
            ),
            "current_queue_length": sum(
                customer.wait_start_time is not None
                and customer.assigned_employee_id is None
                for customer in self.customers
            ),
            "sensor_count": len(self.sensors),
        }

## 3. Initialize the example system

The following data define the illustrative employee, workstation, and sensor populations. Hourly rates, skills, capacities, and sensor specifications are inputs to the example. In this simulation, employee pay rates affect labor cost; employee skills, workstation capacity, and sensor readings are descriptive only.

In [ ]:
print("Initializing the workforce simulation...")

digital_twin = WorkforceDigitalTwin(
    simulation_duration=480.0,
    time_step=5.0,
)

employee_configs = [
    {
        "id": 1,
        "name": "Alice",
        "skills": ["customer_service", "cashier"],
        "hourly_rate": 22,
        "max_hours": 8,
        "satisfaction_score": 0.9,
    },
    {
        "id": 2,
        "name": "Bob",
        "skills": ["customer_service", "stocking"],
        "hourly_rate": 20,
        "max_hours": 8,
        "satisfaction_score": 0.7,
    },
    {
        "id": 3,
        "name": "Charlie",
        "skills": ["cashier", "supervisor"],
        "hourly_rate": 24,
        "max_hours": 6,
        "satisfaction_score": 0.8,
    },
    {
        "id": 4,
        "name": "Diana",
        "skills": ["customer_service", "specialist"],
        "hourly_rate": 21,
        "max_hours": 8,
        "satisfaction_score": 0.85,
    },
    {
        "id": 5,
        "name": "Eve",
        "skills": ["cashier", "stocking"],
        "hourly_rate": 19,
        "max_hours": 8,
        "satisfaction_score": 0.75,
    },
]
digital_twin.initialize_employees(employee_configs)

workstation_configs = [
    {"id": 1, "name": "Service Desk 1", "capacity": 2, "efficiency": 1.0},
    {"id": 2, "name": "Service Desk 2", "capacity": 2, "efficiency": 0.95},
    {"id": 3, "name": "Express Lane", "capacity": 1, "efficiency": 1.1},
    {"id": 4, "name": "Special Services", "capacity": 1, "efficiency": 0.9},
]
digital_twin.initialize_workstations(workstation_configs)

sensor_configs = [
    {
        "id": 1,
        "name": "Entrance Camera",
        "type": "camera",
        "location": "entrance",
        "frequency": 1.0,
        "accuracy": 0.98,
    },
    {
        "id": 2,
        "name": "Queue Sensor",
        "type": "infrared",
        "location": "queue_area",
        "frequency": 0.5,
        "accuracy": 0.95,
    },
    {
        "id": 3,
        "name": "Service Desk Monitor",
        "type": "rfid",
        "location": "service_desk",
        "frequency": 2.0,
        "accuracy": 0.99,
    },
    {
        "id": 4,
        "name": "Employee Badge Reader",
        "type": "rfid",
        "location": "staff_area",
        "frequency": 0.1,
        "accuracy": 1.0,
    },
]
digital_twin.initialize_sensors(sensor_configs)

print("Simulation entities initialized:")
print(f"  Employees: {len(digital_twin.employees)}")
print(f"  Workstations: {len(digital_twin.workstations)}")
print(f"  Sensor descriptions: {len(digital_twin.sensors)}")
print(f"  Base simulation horizon: {digital_twin.simulation_duration:.0f} minutes")
print(f"  Time step: {digital_twin.time_step:.0f} minutes")

## 4. Define candidate schedules

A schedule maps each employee ID to a shift start and end, in minutes from the start of the simulation. The total staff-hours are the sum of shift durations divided by 60. This descriptive total does not impose each employee's `max_hours` as a constraint; the supplied schedules should therefore be interpreted as scenario inputs rather than validated workforce plans.

In [ ]:
def create_test_schedules() -> Dict[str, Dict[int, List[float]]]:
    """Return the fixed schedules used in the scenario comparison."""
    schedules = {
        "conservative": {
            1: [0, 480],
            2: [0, 480],
            3: [120, 420],
            4: [0, 480],
            5: [60, 420],
        },
        "balanced": {
            1: [0, 420],
            2: [60, 480],
            3: [180, 360],
            4: [0, 360],
            5: [120, 480],
        },
        "aggressive": {
            1: [60, 420],
            2: [120, 360],
            3: [180, 300],
            4: [0, 300],
            5: [180, 420],
        },
        "dynamic": {
            1: [0, 240],
            2: [120, 480],
            3: [60, 300],
            4: [240, 480],
            5: [0, 180],
        },
    }
    return schedules


test_schedules = create_test_schedules()

print("Candidate schedules:")
print("=" * 30)
for name, schedule in test_schedules.items():
    total_hours = sum((end - start) / 60 for start, end in schedule.values())
    print(f"{name:12}: {total_hours:.1f} total staff-hours")

print("\nScheduled labor cost:")
print("-" * 25)
for name, schedule in test_schedules.items():
    labor_cost = 0.0
    for employee_id, (start, end) in schedule.items():
        employee = next(
            employee for employee in digital_twin.employees
            if employee.id == employee_id
        )
        labor_cost += ((end - start) / 60) * employee.hourly_rate
    print(f"{name:12}: ${labor_cost:.2f}")

## 5. Run schedule-and-demand scenarios

Each candidate schedule is evaluated under both the variable and peak arrival patterns. The simulation reports one realization for each scenario. The variable pattern uses a time-varying sinusoidal arrival rate with a lower bound; the peak pattern uses a base rate and higher rates during its specified peak intervals. The scenario labels describe these generated demand patterns, not observed or forecast demand.

In [ ]:
arrival_patterns = ["variable", "peak"]

print("Running workforce simulation scenarios...")
print("=" * 50)

start_time = time.time()
scenario_results = digital_twin.run_scenario_analysis(
    test_schedules,
    arrival_patterns,
)
elapsed_time = time.time() - start_time

print(f"\nScenario analysis completed in {elapsed_time:.2f} seconds")
scenario_count = len(test_schedules) * len(arrival_patterns)
print(
    f"Evaluated {len(test_schedules)} schedules × "
    f"{len(arrival_patterns)} arrival patterns = {scenario_count} scenarios"
)

## 6. Compare the results

The table gathers the same measures from every run. Service level is a fraction between 0 and 1; average and peak waits are in minutes; cost is in dollars; throughput is the number of completed customers within the simulation horizon. “Best” labels below identify the highest or lowest observed value for a single metric. They do not establish an overall best schedule because the metrics can trade off, and ties are possible.

In [ ]:
comparison_data = []
for schedule_name in test_schedules:
    for pattern in arrival_patterns:
        scenario_key = f"{schedule_name}_{pattern}"
        results = scenario_results[scenario_key]
        comparison_data.append(
            {
                "schedule": schedule_name,
                "pattern": pattern,
                "service_level": results["service_level"],
                "avg_wait_time": results["avg_wait_time"],
                "peak_wait": results["peak_wait_time"],
                "utilization": results["avg_utilization"],
                "total_cost": results["total_cost"],
                "cost_per_customer": results["cost_per_customer"],
                "throughput": results["throughput"],
                "peak_queue": results["peak_queue_length"],
            }
        )

comparison_df = pd.DataFrame(comparison_data)
key_metrics = [
    "schedule",
    "pattern",
    "service_level",
    "avg_wait_time",
    "total_cost",
    "throughput",
]
print("Scenario comparison:")
print(comparison_df[key_metrics].round(2).to_string(index=False))

print("\nMetric leaders among the simulated scenarios:")
print("-" * 45)
best_service = comparison_df.loc[comparison_df["service_level"].idxmax()]
lowest_cost = comparison_df.loc[comparison_df["total_cost"].idxmin()]
best_throughput = comparison_df.loc[comparison_df["throughput"].idxmax()]
positive_customer_rows = comparison_df[comparison_df["throughput"] > 0]

print(
    "Highest service level: "
    f"{best_service['schedule']} ({best_service['pattern']}) — "
    f"{best_service['service_level']:.2%}"
)
print(
    "Lowest total cost: "
    f"{lowest_cost['schedule']} ({lowest_cost['pattern']}) — "
    f"${lowest_cost['total_cost']:.2f}"
)
print(
    "Highest throughput: "
    f"{best_throughput['schedule']} ({best_throughput['pattern']}) — "
    f"{best_throughput['throughput']:.0f} customers"
)
if not positive_customer_rows.empty:
    best_efficiency = positive_customer_rows.loc[
        positive_customer_rows["cost_per_customer"].idxmin()
    ]
    print(
        "Lowest cost per completed customer: "
        f"{best_efficiency['schedule']} ({best_efficiency['pattern']}) — "
        f"${best_efficiency['cost_per_customer']:.2f} per customer"
    )
else:
    print("Cost per completed customer is undefined because no customers completed service.")

## 7. Visualize scenario trade-offs

The heat maps show schedule-by-pattern comparisons; the scatter plots show how metrics move together; and the grouped bars compare peak queue lengths. Each panel names its units or metric, and the color bars explain the values encoded by color. Patterns in these plots describe the simulated runs and should not be interpreted as statistical significance.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Workforce Simulation Scenario Analysis", fontsize=16, fontweight="bold")
schedule_names = list(test_schedules)

# Heat maps make the schedule and arrival-pattern comparisons easy to scan.
pivot_service = comparison_df.pivot(
    index="schedule", columns="pattern", values="service_level"
).reindex(index=schedule_names, columns=arrival_patterns)
image = axes[0, 0].imshow(
    pivot_service.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1
)
axes[0, 0].set_title("Service Level")
axes[0, 0].set_xlabel("Arrival pattern")
axes[0, 0].set_ylabel("Schedule")
axes[0, 0].set_xticks(range(len(arrival_patterns)))
axes[0, 0].set_xticklabels(arrival_patterns)
axes[0, 0].set_yticks(range(len(schedule_names)))
axes[0, 0].set_yticklabels(schedule_names)
for row in range(len(schedule_names)):
    for column in range(len(arrival_patterns)):
        axes[0, 0].text(
            column, row, f"{pivot_service.iloc[row, column]:.2f}",
            ha="center", va="center", color="black"
        )
fig.colorbar(image, ax=axes[0, 0], label="Satisfied share (0–1)")

pivot_cost = comparison_df.pivot(
    index="schedule", columns="pattern", values="total_cost"
).reindex(index=schedule_names, columns=arrival_patterns)
image = axes[0, 1].imshow(pivot_cost.values, cmap="RdYlBu_r", aspect="auto")
axes[0, 1].set_title("Total Operating Cost")
axes[0, 1].set_xlabel("Arrival pattern")
axes[0, 1].set_ylabel("Schedule")
axes[0, 1].set_xticks(range(len(arrival_patterns)))
axes[0, 1].set_xticklabels(arrival_patterns)
axes[0, 1].set_yticks(range(len(schedule_names)))
axes[0, 1].set_yticklabels(schedule_names)
for row in range(len(schedule_names)):
    for column in range(len(arrival_patterns)):
        axes[0, 1].text(
            column, row, f"${pivot_cost.iloc[row, column]:.0f}",
            ha="center", va="center", color="black"
        )
fig.colorbar(image, ax=axes[0, 1], label="Cost (dollars)")

pivot_wait = comparison_df.pivot(
    index="schedule", columns="pattern", values="avg_wait_time"
).reindex(index=schedule_names, columns=arrival_patterns)
image = axes[0, 2].imshow(pivot_wait.values, cmap="RdYlBu_r", aspect="auto")
axes[0, 2].set_title("Average Waiting Time")
axes[0, 2].set_xlabel("Arrival pattern")
axes[0, 2].set_ylabel("Schedule")
axes[0, 2].set_xticks(range(len(arrival_patterns)))
axes[0, 2].set_xticklabels(arrival_patterns)
axes[0, 2].set_yticks(range(len(schedule_names)))
axes[0, 2].set_yticklabels(schedule_names)
for row in range(len(schedule_names)):
    for column in range(len(arrival_patterns)):
        axes[0, 2].text(
            column, row, f"{pivot_wait.iloc[row, column]:.1f}",
            ha="center", va="center", color="black"
        )
fig.colorbar(image, ax=axes[0, 2], label="Minutes")

colors = plt.cm.Set1(np.linspace(0, 1, len(schedule_names)))
markers = {"variable": "o", "peak": "s"}
for index, schedule_name in enumerate(schedule_names):
    for pattern in arrival_patterns:
        row = comparison_df[
            (comparison_df["schedule"] == schedule_name)
            & (comparison_df["pattern"] == pattern)
        ]
        axes[1, 0].scatter(
            row["total_cost"],
            row["throughput"],
            color=colors[index],
            marker=markers[pattern],
            s=90,
            alpha=0.8,
            label=f"{schedule_name} ({pattern})",
        )
axes[1, 0].set_xlabel("Total cost (dollars)")
axes[1, 0].set_ylabel("Completed customers")
axes[1, 0].set_title("Cost and Throughput")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend(fontsize=7, loc="best")

for index, schedule_name in enumerate(schedule_names):
    for pattern in arrival_patterns:
        row = comparison_df[
            (comparison_df["schedule"] == schedule_name)
            & (comparison_df["pattern"] == pattern)
        ]
        axes[1, 1].scatter(
            row["utilization"],
            row["service_level"],
            color=colors[index],
            marker=markers[pattern],
            s=90,
            alpha=0.8,
        )
axes[1, 1].set_xlabel("Employee utilization (share)")
axes[1, 1].set_ylabel("Service level (share)")
axes[1, 1].set_title("Utilization and Service Level")
axes[1, 1].grid(True, alpha=0.3)

positions = np.arange(len(schedule_names))
bar_width = 0.35
for pattern_index, pattern in enumerate(arrival_patterns):
    pattern_rows = comparison_df[comparison_df["pattern"] == pattern].set_index("schedule")
    peak_queues = [pattern_rows.loc[name, "peak_queue"] for name in schedule_names]
    axes[1, 2].bar(
        positions + pattern_index * bar_width,
        peak_queues,
        bar_width,
        label=pattern,
        alpha=0.85,
    )
axes[1, 2].set_xlabel("Schedule")
axes[1, 2].set_ylabel("Customers waiting")
axes[1, 2].set_title("Peak Queue Length")
axes[1, 2].set_xticks(positions + bar_width / 2)
axes[1, 2].set_xticklabels(schedule_names, rotation=35, ha="right")
axes[1, 2].legend(title="Arrival pattern")
axes[1, 2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Inspect simulated status and illustrative events

The status report describes the state left by the most recently completed run. Its synchronization flags are false because this example has no live data connections. The event list below is illustrative text: it is not injected into the simulation, and the recommendations are not computed by a decision-support algorithm. Keeping this distinction explicit prevents a demonstration of an interface idea from being mistaken for a measured operational result.

In [ ]:
status = digital_twin.get_real_time_status()

print("Simulated system status:")
for key, value in status.items():
    if key == "sync_status":
        print("Data connection status:")
        for component, connected in value.items():
            label = "connected" if connected else "not connected"
            print(f"  {component.replace('_', ' ').title()}: {label}")
    else:
        print(f"{key.replace('_', ' ').title()}: {value}")

illustrative_events = [
    {
        "time": 120,
        "type": "employee_break",
        "detail": "Employee 3 takes a 15-minute break.",
    },
    {
        "time": 180,
        "type": "demand_spike",
        "detail": "Demand rises by 50% for 30 minutes.",
    },
    {
        "time": 240,
        "type": "workstation_issue",
        "detail": "Workstation 2 loses 30% of its efficiency.",
    },
    {
        "time": 300,
        "type": "customer_complaint",
        "detail": "A high-severity complaint is recorded.",
    },
    {
        "time": 360,
        "type": "shift_change",
        "detail": "Employees 1 and 4 are scheduled to finish.",
    },
]

print("\nIllustrative event messages (not applied to the simulation):")
for event in illustrative_events:
    print(
        f"T+{event['time']:3d} min — "
        f"{event['type'].replace('_', ' ').title()}: {event['detail']}"
    )

illustrative_recommendations = [
    "Consider assigning a qualified employee to cover a planned break.",
    "Review staffing coverage during a demand increase.",
    "Check whether customers can be served at another workstation.",
    "Route a complaint to an appropriate staff member.",
    "Plan a handover when scheduled shifts end.",
]
print("\nIllustrative operational questions:")
for number, recommendation in enumerate(illustrative_recommendations, start=1):
    print(f"{number}. {recommendation}")

## 9. What-if scenario analysis

What-if analysis asks how the modeled outcomes change when scenario inputs change. The demand-surge case passes a multiplier to the arrival generator; the staff-shortage case removes one employee from the schedule. The extended-hours case lengthens the customer-arrival and simulation horizon while keeping the conservative schedule unchanged. Consequently, the additional period has no scheduled shift coverage, which is a deliberate illustration of how an operating-horizon change can create a staffing gap.

These are separate stochastic runs, so differences reflect both scenario inputs and random variation. The printed percentage changes are calculated from the run results; they are not forecast guarantees. The recommendation list is a set of discussion prompts, not a model-generated optimum.

In [ ]:
what_if_scenarios = {
    "base_case": {
        "description": "Balanced schedule with variable arrivals",
        "schedule": test_schedules["balanced"],
        "pattern": "variable",
    },
    "demand_surge": {
        "description": "A 25% increase in the generated peak-arrival rate",
        "schedule": test_schedules["balanced"],
        "pattern": "peak",
        "demand_multiplier": 1.25,
    },
    "staff_shortage": {
        "description": "Employee 5 is unavailable during variable arrivals",
        "schedule": {
            employee_id: shift
            for employee_id, shift in test_schedules["balanced"].items()
            if employee_id != 5
        },
        "pattern": "variable",
    },
    "extended_hours": {
        "description": "A 120-minute extension with the original shift schedule",
        "schedule": test_schedules["conservative"],
        "pattern": "variable",
        "simulation_duration": 600.0,
    },
}

what_if_results = {}
for scenario_name, scenario_config in what_if_scenarios.items():
    print(f"\nScenario: {scenario_name}")
    print(f"Description: {scenario_config['description']}")
    results = digital_twin.run_simulation(
        schedule=scenario_config["schedule"],
        arrival_pattern=scenario_config["pattern"],
        demand_multiplier=scenario_config.get("demand_multiplier", 1.0),
        simulation_duration=scenario_config.get("simulation_duration"),
    )
    what_if_results[scenario_name] = results
    print(f"  Service level: {results['service_level']:.1%}")
    print(f"  Average wait: {results['avg_wait_time']:.1f} minutes")
    print(f"  Total cost: ${results['total_cost']:.2f}")
    print(f"  Completed customers: {results['throughput']:.0f}")

base_results = what_if_results["base_case"]
print("\nChange relative to the base-case run:")
print("-" * 45)
for scenario_name, results in what_if_results.items():
    if scenario_name == "base_case":
        continue

    service_change = (results["service_level"] - base_results["service_level"]) * 100
    if base_results["total_cost"]:
        cost_change = (
            (results["total_cost"] - base_results["total_cost"])
            / base_results["total_cost"]
            * 100
        )
        cost_text = f"{cost_change:+.1f}%"
    else:
        cost_text = "undefined (base-case cost is zero)"
    wait_change = results["avg_wait_time"] - base_results["avg_wait_time"]
    print(
        f"{scenario_name:15}: service level {service_change:+.1f} percentage points, "
        f"cost {cost_text}, average wait {wait_change:+.1f} minutes"
    )

print("\nQuestions for operational review:")
print("- Which service and cost measures should guide a schedule decision?")
print("- How much scheduled coverage is needed during the highest-demand periods?")
print("- What cross-training or contingency coverage would be feasible?")
print("- Which additional operational data would be needed to validate this model?")

## 10. Interpretation and appropriate use

The simulation provides a structured way to compare specified schedules under stated demand assumptions. A schedule can have lower cost yet also produce longer waits or lower service, so the metric leaders should be read together rather than treated as a single overall ranking. If two scenarios have equal reported values, both share that metric's lead.

This model's value is its explicit sequence of arrival generation, resource assignment, service completion, and metric aggregation. Its limits are equally important: arrivals and service times are synthetic; skills, workstation capacity, breaks, fatigue, and sensor accuracy do not affect the service logic; demand and staffing are not optimized; and the results come from single stochastic realizations. In an operational setting, model assumptions would need to be calibrated and results validated against relevant observed data before they could support decisions.

### When this simulation structure can be useful
- Comparing a small set of proposed schedules under clearly defined demand scenarios.
- Identifying which assumptions or performance measures most influence a planning discussion.
- Communicating potential staffing and service trade-offs to stakeholders.
- Developing a more detailed simulation by adding validated operational rules and data.

### Possible extensions
Natural extensions include modeling breaks, skill-based assignments, workstation capacity, employee-hour limits, and repeated simulation replications. Those additions would require explicit assumptions and validation; they are not implemented in this notebook.